<a href="https://colab.research.google.com/github/ayy77v/Cuda-Gemm/blob/main/Cuda_Gemm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%%writefile matmul_kernel1.cu
#include <cstdio>
#include <cstdlib>
#include <cmath>
#include <cuda_runtime.h>

#define CEIL_DIV(M, N) (((M) + (N) - 1) / (N))

__global__ void sgemm_naive(int M, int N, int K, float alpha,
                            const float *A, const float *B,
                            float beta, float *C) {
    const uint x = blockIdx.x * blockDim.x + threadIdx.x;
    const uint y = blockIdx.y * blockDim.y + threadIdx.y;

    if (x < M && y < N) {
        float tmp = 0.0f;
        for (int i = 0; i < K; ++i) {
            tmp += A[x * K + i] * B[i * N + y];
        }
        C[x * N + y] = alpha * tmp + beta * C[x * N + y];
    }
}

int main() {
    int M = 1024, N = 1024, K = 1024;
    size_t size_A = M * K * sizeof(float);
    size_t size_B = K * N * sizeof(float);
    size_t size_C = M * N * sizeof(float);

    float *h_A = (float*)malloc(size_A);
    float *h_B = (float*)malloc(size_B);
    float *h_C = (float*)malloc(size_C);

    for (int i = 0; i < M * K; ++i) h_A[i] = 1.0f;
    for (int i = 0; i < K * N; ++i) h_B[i] = 2.0f;

    float *d_A, *d_B, *d_C;
    cudaMalloc(&d_A, size_A);
    cudaMalloc(&d_B, size_B);
    cudaMalloc(&d_C, size_C);

    cudaMemcpy(d_A, h_A, size_A, cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, h_B, size_B, cudaMemcpyHostToDevice);

    dim3 blockDim(32, 32, 1);
    dim3 gridDim(CEIL_DIV(M, 32), CEIL_DIV(N, 32), 1);

    sgemm_naive<<<gridDim, blockDim>>>(M, N, K, 1.0f, d_A, d_B, 0.0f, d_C);
    cudaDeviceSynchronize();

    cudaMemcpy(h_C, d_C, size_C, cudaMemcpyDeviceToHost);

    printf("C[0] = %f (Expected: %f)\n", h_C[0], (float)K * 2.0f);

    cudaFree(d_A); cudaFree(d_B); cudaFree(d_C);
    free(h_A); free(h_B); free(h_C);
    return 0;
}

Writing matmul_kernel1.cu


In [ ]:
!nvcc -O3 -lcublas matmul_kernel1.cu -o matmul1
!./matmul1

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
C[0] = 2048.000000 (Expected: 2048.000000)


In [ ]:
%%writefile matmul_kernel2.cu
#include <cstdio>
#include <cstdlib>
#include <cmath>
#include <cuda_runtime.h>

#define CEIL_DIV(M, N) (((M) + (N) - 1) / (N))
#define BLOCKSIZE 32

__global__ void sgemm_coalescing(int M, int N, int K, float alpha,
                                 const float *A, const float *B,
                                 float beta, float *C) {
    const int cRow = blockIdx.x * BLOCKSIZE + (threadIdx.x / BLOCKSIZE);
    const int cCol = blockIdx.y * BLOCKSIZE + (threadIdx.x % BLOCKSIZE);

    if (cRow < M && cCol < N) {
        float tmp = 0.0f;
        for (int i = 0; i < K; ++i) {
            tmp += A[cRow * K + i] * B[i * N + cCol];
        }
        C[cRow * N + cCol] = alpha * tmp + beta * C[cRow * N + cCol];
    }
}

int main() {
    int M = 2048, N = 2048, K = 2048;
    size_t size_A = M * K * sizeof(float);
    size_t size_B = K * N * sizeof(float);
    size_t size_C = M * N * sizeof(float);

    float *h_A = (float*)malloc(size_A);
    float *h_B = (float*)malloc(size_B);
    float *h_C = (float*)malloc(size_C);

    for (int i = 0; i < M * K; ++i) h_A[i] = 1.0f;
    for (int i = 0; i < K * N; ++i) h_B[i] = 2.0f;

    float *d_A, *d_B, *d_C;
    cudaMalloc(&d_A, size_A);
    cudaMalloc(&d_B, size_B);
    cudaMalloc(&d_C, size_C);

    cudaMemcpy(d_A, h_A, size_A, cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, h_B, size_B, cudaMemcpyHostToDevice);

    // Launch configuration: 1D Block (1024 threads)
    dim3 blockDim(BLOCKSIZE * BLOCKSIZE, 1, 1);
    dim3 gridDim(CEIL_DIV(M, BLOCKSIZE), CEIL_DIV(N, BLOCKSIZE), 1);

    // 建立計時 Event
    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    // Warm up
    sgemm_coalescing<<<gridDim, blockDim>>>(M, N, K, 1.0f, d_A, d_B, 0.0f, d_C);
    cudaDeviceSynchronize();

    // 測量執行時間
    cudaEventRecord(start);
    sgemm_coalescing<<<gridDim, blockDim>>>(M, N, K, 1.0f, d_A, d_B, 0.0f, d_C);
    cudaEventRecord(stop);
    cudaEventSynchronize(stop);

    float milliseconds = 0;
    cudaEventElapsedTime(&milliseconds, start, stop);

    // 驗證輸出
    cudaMemcpy(h_C, d_C, size_C, cudaMemcpyDeviceToHost);
    printf("Verification: C[0] = %f (Expected: %f)\n", h_C[0], (float)K * 2.0f);

    // 計算 GFLOPS: 2 * M * N * K FLOPs
    double flops = 2.0 * (double)M * (double)N * (double)K;
    double gflops = (flops / (milliseconds / 1000.0)) / 1e9;
    printf("Execution Time: %.2f ms | Performance: %.2f GFLOPS\n", milliseconds, gflops);

    cudaFree(d_A); cudaFree(d_B); cudaFree(d_C);
    free(h_A); free(h_B); free(h_C);
    return 0;
}

Writing matmul_kernel2.cu


In [ ]:
!nvcc -O3 -arch=sm_75 -lcublas matmul_kernel2.cu -o matmul2
!./matmul2

Verification: C[0] = 4096.000000 (Expected: 4096.000000)
Execution Time: 4.94 ms | Performance: 3480.75 GFLOPS


In [ ]:
  %%writefile matmul_kernel3.cu
  #include <cstdio>
  #include <cstdlib>
  #include <cmath>
  #include <cuda_runtime.h>

  #define CEIL_DIV(M, N) (((M) + (N) - 1) / (N))
  #define BLOCKSIZE 32

  __global__ void sgemm_shared_mem(int M, int N, int K, float alpha,
                                  const float *A, const float *B,
                                  float beta, float *C) {
      // 該 block 負責計算 C 的起點
      const uint cRow = blockIdx.x;
      const uint cCol = blockIdx.y;

      // 分配 Shared Memory (32x32 = 1024 floats, 約 4KB)
      __shared__ float As[BLOCKSIZE * BLOCKSIZE];
      __shared__ float Bs[BLOCKSIZE * BLOCKSIZE];

      // 當前 thread 在 block 內的 (row, col)
      const uint threadCol = threadIdx.x % BLOCKSIZE;
      const uint threadRow = threadIdx.x / BLOCKSIZE;

      // 將指針移到當前 Block 負責的子區塊起始位置
      A += cRow * BLOCKSIZE * K;
      B += cCol * BLOCKSIZE;
      C += (cRow * BLOCKSIZE + threadRow) * N + (cCol * BLOCKSIZE + threadCol);

      float tmp = 0.0f;

      // 沿著 K 維度以 BLOCKSIZE 為步長進行分塊遍歷
      for (int bkIdx = 0; bkIdx < K; bkIdx += BLOCKSIZE) {
          // 協同載入 Global Memory 到 Shared Memory
          As[threadRow * BLOCKSIZE + threadCol] = A[threadRow * K + threadCol];
          Bs[threadRow * BLOCKSIZE + threadCol] = B[threadRow * N + threadCol];

          // 同步，確保整個 Tile 都已載入到 Shared Memory
          __syncthreads();

          // 前移 Global 矩陣指針，為下一輪迭代做準備
          A += BLOCKSIZE;
          B += BLOCKSIZE * N;

          // 從 Shared Memory 讀取並計算乘積
          for (int dotIdx = 0; dotIdx < BLOCKSIZE; ++dotIdx) {
              tmp += As[threadRow * BLOCKSIZE + dotIdx] * Bs[dotIdx * BLOCKSIZE + threadCol];
          }

          // 同步，確保當前輪次計算完畢後才覆蓋 Shared Memory
          __syncthreads();
      }

      // 寫回計算結果
      *C = alpha * tmp + beta * *C;
  }

  int main() {
      int M = 2048, N = 2048, K = 2048;
      size_t size_A = M * K * sizeof(float);
      size_t size_B = K * N * sizeof(float);
      size_t size_C = M * N * sizeof(float);

      float *h_A = (float*)malloc(size_A);
      float *h_B = (float*)malloc(size_B);
      float *h_C = (float*)malloc(size_C);

      for (int i = 0; i < M * K; ++i) h_A[i] = 1.0f;
      for (int i = 0; i < K * N; ++i) h_B[i] = 2.0f;

      float *d_A, *d_B, *d_C;
      cudaMalloc(&d_A, size_A);
      cudaMalloc(&d_B, size_B);
      cudaMalloc(&d_C, size_C);

      cudaMemcpy(d_A, h_A, size_A, cudaMemcpyHostToDevice);
      cudaMemcpy(d_B, h_B, size_B, cudaMemcpyHostToDevice);

      dim3 blockDim(BLOCKSIZE * BLOCKSIZE, 1, 1);
      dim3 gridDim(CEIL_DIV(M, BLOCKSIZE), CEIL_DIV(N, BLOCKSIZE), 1);

      cudaEvent_t start, stop;
      cudaEventCreate(&start);
      cudaEventCreate(&stop);

      // Warm up
      sgemm_shared_mem<<<gridDim, blockDim>>>(M, N, K, 1.0f, d_A, d_B, 0.0f, d_C);
      cudaDeviceSynchronize();

      // 計時
      cudaEventRecord(start);
      sgemm_shared_mem<<<gridDim, blockDim>>>(M, N, K, 1.0f, d_A, d_B, 0.0f, d_C);
      cudaEventRecord(stop);
      cudaEventSynchronize(stop);

      float milliseconds = 0;
      cudaEventElapsedTime(&milliseconds, start, stop);

      cudaMemcpy(h_C, d_C, size_C, cudaMemcpyDeviceToHost);
      printf("Verification: C[0] = %f (Expected: %f)\n", h_C[0], (float)K * 2.0f);

      double flops = 2.0 * (double)M * (double)N * (double)K;
      double gflops = (flops / (milliseconds / 1000.0)) / 1e9;
      printf("Execution Time: %.2f ms | Performance: %.2f GFLOPS\n", milliseconds, gflops);

      cudaFree(d_A); cudaFree(d_B); cudaFree(d_C);
      free(h_A); free(h_B); free(h_C);
      return 0;
  }

Writing matmul_kernel3.cu


In [ ]:
!nvcc -O3 -arch=sm_75 -lcublas matmul_kernel3.cu -o matmul3
!./matmul3

Verification: C[0] = 4096.000000 (Expected: 4096.000000)
Execution Time: 3.31 ms | Performance: 5184.55 GFLOPS


In [ ]:
%%writefile matmul_kernel4.cu
#include <cstdio>
#include <cstdlib>
#include <cmath>
#include <cuda_runtime.h>

#define CEIL_DIV(M, N) (((M) + (N) - 1) / (N))

// 定義分塊大小
const int BM = 64;
const int BN = 64;
const int BK = 8;
const int TM = 8;

__global__ void sgemm_1d_blocktiling(int M, int N, int K, float alpha,
                                     const float *A, const float *B,
                                     float beta, float *C) {
    const uint cRow = blockIdx.x;
    const uint cCol = blockIdx.y;

    // Block 內有 (BM * BN) / TM = 512 個 threads
    const uint totalResultsBlocktile = BM * BN;
    const uint numThreadsBlocktile = totalResultsBlocktile / TM;

    // 計算當前 thread 在 Block 內的相對位置
    const uint threadCol = threadIdx.x % BN;
    const uint threadRow = threadIdx.x / BN;

    // 分配 Shared Memory
    __shared__ float As[BM * BK];
    __shared__ float Bs[BK * BN];

    // 將矩陣指標移至該 Block 的起始位置
    A += cRow * BM * K;
    B += cCol * BN;
    C += cRow * BM * N + cCol * BN;

    // 計算協同載入 Shared Memory 時的 thread 索引
    const uint innerRowA = threadIdx.x / BK;
    const uint innerColA = threadIdx.x % BK;
    const uint strideA = numThreadsBlocktile / BK;

    const uint innerRowB = threadIdx.x / BN;
    const uint innerColB = threadIdx.x % BN;
    const uint strideB = numThreadsBlocktile / BN;

    // 每個 thread 在暫存器 (Registers) 內維護 TM 個結果
    float threadResults[TM] = {0.0f};

    // 主循環：沿著 K 軸滑動 BK
    for (int bkIdx = 0; bkIdx < K; bkIdx += BK) {
        // 協同載入 A 矩陣到 As (BM x BK)
        for (uint offset = 0; offset < BM; offset += strideA) {
            As[(innerRowA + offset) * BK + innerColA] = A[(innerRowA + offset) * K + innerColA];
        }
        // 協同載入 B 矩陣到 Bs (BK x BN)
        for (uint offset = 0; offset < BK; offset += strideB) {
            Bs[(innerRowB + offset) * BN + innerColB] = B[(innerRowB + offset) * N + innerColB];
        }
        __syncthreads();

        // 推進 Global 指標
        A += BK;
        B += BK * N;

        // 計算局部內積
        for (uint dotIdx = 0; dotIdx < BK; ++dotIdx) {
            // 從 Shared Memory 載入 1 個 B 元素到暫存器
            float B_val = Bs[dotIdx * BN + threadCol];
            // 重複利用該 B 元素與 TM 個 A 元素做乘累加
            for (uint resIdx = 0; resIdx < TM; ++resIdx) {
                threadResults[resIdx] += As[(threadRow * TM + resIdx) * BK + dotIdx] * B_val;
            }
        }
        __syncthreads();
    }

    // 將暫存器結果寫回 Global Memory
    for (uint resIdx = 0; resIdx < TM; ++resIdx) {
        int r = threadRow * TM + resIdx;
        int c = threadCol;
        if ((cRow * BM + r) < M && (cCol * BN + c) < N) {
            C[r * N + c] = alpha * threadResults[resIdx] + beta * C[r * N + c];
        }
    }
}

int main() {
    int M = 2048, N = 2048, K = 2048;
    size_t size_A = M * K * sizeof(float);
    size_t size_B = K * N * sizeof(float);
    size_t size_C = M * N * sizeof(float);

    float *h_A = (float*)malloc(size_A);
    float *h_B = (float*)malloc(size_B);
    float *h_C = (float*)malloc(size_C);

    for (int i = 0; i < M * K; ++i) h_A[i] = 1.0f;
    for (int i = 0; i < K * N; ++i) h_B[i] = 2.0f;

    float *d_A, *d_B, *d_C;
    cudaMalloc(&d_A, size_A);
    cudaMalloc(&d_B, size_B);
    cudaMalloc(&d_C, size_C);

    cudaMemcpy(d_A, h_A, size_A, cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, h_B, size_B, cudaMemcpyHostToDevice);

    // 512 threads per block
    dim3 blockDim((BM * BN) / TM, 1, 1);
    dim3 gridDim(CEIL_DIV(M, BM), CEIL_DIV(N, BN), 1);

    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    // Warm up
    sgemm_1d_blocktiling<<<gridDim, blockDim>>>(M, N, K, 1.0f, d_A, d_B, 0.0f, d_C);
    cudaDeviceSynchronize();

    // 計時
    cudaEventRecord(start);
    sgemm_1d_blocktiling<<<gridDim, blockDim>>>(M, N, K, 1.0f, d_A, d_B, 0.0f, d_C);
    cudaEventRecord(stop);
    cudaEventSynchronize(stop);

    float milliseconds = 0;
    cudaEventElapsedTime(&milliseconds, start, stop);

    cudaMemcpy(h_C, d_C, size_C, cudaMemcpyDeviceToHost);
    printf("Verification: C[0] = %f (Expected: %f)\n", h_C[0], (float)K * 2.0f);

    double flops = 2.0 * (double)M * (double)N * (double)K;
    double gflops = (flops / (milliseconds / 1000.0)) / 1e9;
    printf("Execution Time: %.2f ms | Performance: %.2f GFLOPS\n", milliseconds, gflops);

    cudaFree(d_A); cudaFree(d_B); cudaFree(d_C);
    free(h_A); free(h_B); free(h_C);
    return 0;
}

Writing matmul_kernel4.cu


In [ ]:
!nvcc -O3 -arch=sm_75 -lcublas matmul_kernel4.cu -o matmul4
!./matmul4

Verification: C[0] = 4096.000000 (Expected: 4096.000000)
Execution Time: 1.77 ms | Performance: 9725.92 GFLOPS


In [ ]:
%%writefile matmul_kernel5.cu
#include <cstdio>
#include <cstdlib>
#include <cmath>
#include <cuda_runtime.h>

#define CEIL_DIV(M, N) (((M) + (N) - 1) / (N))

// 2D Block Tiling 參數
const int BM = 128;
const int BN = 128;
const int BK = 8;
const int TM = 8;
const int TN = 8;

__global__ void sgemm_2d_blocktiling(int M, int N, int K, float alpha,
                                     const float *A, const float *B,
                                     float beta, float *C) {
    const uint cRow = blockIdx.x;
    const uint cCol = blockIdx.y;

    // 總 threads = (128 * 128) / (8 * 8) = 256
    const uint numThreadsBlocktile = (BM * BN) / (TM * TN);

    // 每個 Block 計算 BM x BN 的區域
    // Block 內部的 2D thread 佈局 (以 8x8 Tile 為單位)
    const uint threadCol = threadIdx.x % (BN / TN); // 0 ~ 15
    const uint threadRow = threadIdx.x / (BN / TN); // 0 ~ 15

    __shared__ float As[BM * BK];
    __shared__ float Bs[BK * BN];

    // 指標移至該 Block 的起點
    A += cRow * BM * K;
    B += cCol * BN;
    C += cRow * BM * N + cCol * BN;

    // 協同載入 A 到 Shared Memory 的索引步長
    const uint innerRowA = threadIdx.x / BK;
    const uint innerColA = threadIdx.x % BK;
    const uint strideA = numThreadsBlocktile / BK;

    // 協同載入 B 到 Shared Memory 的索引步長
    const uint innerRowB = threadIdx.x / BN;
    const uint innerColB = threadIdx.x % BN;
    const uint strideB = numThreadsBlocktile / BN;

    // 暫存器配置
    float threadResults[TM * TN] = {0.0f};
    float regM[TM] = {0.0f};
    float regN[TN] = {0.0f};

    // 主循環：沿著 K 維度步進 BK
    for (int bkIdx = 0; bkIdx < K; bkIdx += BK) {
        // 載入 A 到 As (BM x BK)
        for (uint offset = 0; offset < BM; offset += strideA) {
            As[(innerRowA + offset) * BK + innerColA] = A[(innerRowA + offset) * K + innerColA];
        }
        // 載入 B 到 Bs (BK x BN)
        for (uint offset = 0; offset < BK; offset += strideB) {
            Bs[(innerRowB + offset) * BN + innerColB] = B[(innerRowB + offset) * N + innerColB];
        }
        __syncthreads();

        A += BK;
        B += BK * N;

        // 計算 BM x BN 區域內的局部外積
        for (uint dotIdx = 0; dotIdx < BK; ++dotIdx) {
            // 將 As, Bs 資料預先讀取到暫存器中
            for (uint i = 0; i < TM; ++i) {
                regM[i] = As[(threadRow * TM + i) * BK + dotIdx];
            }
            for (uint i = 0; i < TN; ++i) {
                regN[i] = Bs[dotIdx * BN + (threadCol * TN + i)];
            }

            // 暫存器級別的外積計算 (8x8 = 64 次 FMA 運算)
            for (uint resIdxM = 0; resIdxM < TM; ++resIdxM) {
                for (uint resIdxN = 0; resIdxN < TN; ++resIdxN) {
                    threadResults[resIdxM * TN + resIdxN] += regM[resIdxM] * regN[resIdxN];
                }
            }
        }
        __syncthreads();
    }

    // 將 8x8 的結果寫回 Global Memory
    for (uint resIdxM = 0; resIdxM < TM; ++resIdxM) {
        for (uint resIdxN = 0; resIdxN < TN; ++resIdxN) {
            int r = threadRow * TM + resIdxM;
            int c = threadCol * TN + resIdxN;
            if ((cRow * BM + r) < M && (cCol * BN + c) < N) {
                C[r * N + c] = alpha * threadResults[resIdxM * TN + resIdxN] + beta * C[r * N + c];
            }
        }
    }
}

int main() {
    int M = 2048, N = 2048, K = 2048;
    size_t size_A = M * K * sizeof(float);
    size_t size_B = K * N * sizeof(float);
    size_t size_C = M * N * sizeof(float);

    float *h_A = (float*)malloc(size_A);
    float *h_B = (float*)malloc(size_B);
    float *h_C = (float*)malloc(size_C);

    for (int i = 0; i < M * K; ++i) h_A[i] = 1.0f;
    for (int i = 0; i < K * N; ++i) h_B[i] = 2.0f;

    float *d_A, *d_B, *d_C;
    cudaMalloc(&d_A, size_A);
    cudaMalloc(&d_B, size_B);
    cudaMalloc(&d_C, size_C);

    cudaMemcpy(d_A, h_A, size_A, cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, h_B, size_B, cudaMemcpyHostToDevice);

    // 256 threads per block
    dim3 blockDim((BM * BN) / (TM * TN), 1, 1);
    dim3 gridDim(CEIL_DIV(M, BM), CEIL_DIV(N, BN), 1);

    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    // Warm up
    sgemm_2d_blocktiling<<<gridDim, blockDim>>>(M, N, K, 1.0f, d_A, d_B, 0.0f, d_C);
    cudaDeviceSynchronize();

    // 計時
    cudaEventRecord(start);
    sgemm_2d_blocktiling<<<gridDim, blockDim>>>(M, N, K, 1.0f, d_A, d_B, 0.0f, d_C);
    cudaEventRecord(stop);
    cudaEventSynchronize(stop);

    float milliseconds = 0;
    cudaEventElapsedTime(&milliseconds, start, stop);

    cudaMemcpy(h_C, d_C, size_C, cudaMemcpyDeviceToHost);
    printf("Verification: C[0] = %f (Expected: %f)\n", h_C[0], (float)K * 2.0f);

    double flops = 2.0 * (double)M * (double)N * (double)K;
    double gflops = (flops / (milliseconds / 1000.0)) / 1e9;
    printf("Execution Time: %.2f ms | Performance: %.2f GFLOPS\n", milliseconds, gflops);

    cudaFree(d_A); cudaFree(d_B); cudaFree(d_C);
    free(h_A); free(h_B); free(h_C);
    return 0;
}

Writing matmul_kernel5.cu


In [ ]:
!nvcc -O3 -arch=sm_75 -lcublas matmul_kernel5.cu -o matmul5
!./matmul5

Verification: C[0] = 4096.000000 (Expected: 4096.000000)
Execution Time: 1.56 ms | Performance: 11023.14 GFLOPS


In [ ]:
%%writefile matmul_kernel6.cu
#include <cstdio>
#include <cstdlib>
#include <cmath>
#include <cuda_runtime.h>

#define CEIL_DIV(M, N) (((M) + (N) - 1) / (N))

const int BM = 128;
const int BN = 128;
const int BK = 8;
const int TM = 8;
const int TN = 8;

__global__ void sgemm_vectorized(int M, int N, int K, float alpha,
                                 const float *A, const float *B,
                                 float beta, float *C) {
    const uint cRow = blockIdx.x;
    const uint cCol = blockIdx.y;

    const uint totalResultsBlocktile = BM * BN;
    // 總 threads = (128 * 128) / (8 * 8) = 256
    const uint numThreadsBlocktile = totalResultsBlocktile / (TM * TN);

    const uint threadCol = threadIdx.x % (BN / TN); // 0 ~ 15
    const uint threadRow = threadIdx.x / (BN / TN); // 0 ~ 15

    __shared__ float As[BM * BK];
    __shared__ float Bs[BK * BN];

    A += cRow * BM * K;
    B += cCol * BN;
    C += cRow * BM * N + cCol * BN;

    // 向量化載入 A: 每次載入 float4 (4 floats)
    // A 是 BM x BK (128 x 8)，每 row 有 8 floats = 2 個 float4
    const uint innerRowA = threadIdx.x / (BK / 4);
    const uint innerColA = threadIdx.x % (BK / 4);
    const uint strideA = numThreadsBlocktile / (BK / 4);

    // 向量化載入 B: 每次載入 float4 (4 floats)
    // B 是 BK x BN (8 x 128)，每 row 有 128 floats = 32 個 float4
    const uint innerRowB = threadIdx.x / (BN / 4);
    const uint innerColB = threadIdx.x % (BN / 4);
    const uint strideB = numThreadsBlocktile / (BN / 4);

    // 暫存器配置
    float threadResults[TM * TN] = {0.0f};
    float regM[TM] = {0.0f};
    float regN[TN] = {0.0f};

    // 主循環：沿著 K 維度步進 BK
    for (int bkIdx = 0; bkIdx < K; bkIdx += BK) {
        // 使用 float4 向量化載入 A
        for (uint offset = 0; offset < BM; offset += strideA) {
            float4 tmp = reinterpret_cast<const float4*>(
                &A[(innerRowA + offset) * K + innerColA * 4]
            )[0];
            // 轉存到 Shared Memory
            As[(innerRowA + offset) * BK + innerColA * 4 + 0] = tmp.x;
            As[(innerRowA + offset) * BK + innerColA * 4 + 1] = tmp.y;
            As[(innerRowA + offset) * BK + innerColA * 4 + 2] = tmp.z;
            As[(innerRowA + offset) * BK + innerColA * 4 + 3] = tmp.w;
        }

        // 使用 float4 向量化載入 B
        for (uint offset = 0; offset < BK; offset += strideB) {
            reinterpret_cast<float4*>(
                &Bs[(innerRowB + offset) * BN + innerColB * 4]
            )[0] = reinterpret_cast<const float4*>(
                &B[(innerRowB + offset) * N + innerColB * 4]
            )[0];
        }
        __syncthreads();

        A += BK;
        B += BK * N;

        // 計算外積
        for (uint dotIdx = 0; dotIdx < BK; ++dotIdx) {
            for (uint i = 0; i < TM; ++i) {
                regM[i] = As[(threadRow * TM + i) * BK + dotIdx];
            }
            for (uint i = 0; i < TN; ++i) {
                regN[i] = Bs[dotIdx * BN + (threadCol * TN + i)];
            }

            for (uint resIdxM = 0; resIdxM < TM; ++resIdxM) {
                for (uint resIdxN = 0; resIdxN < TN; ++resIdxN) {
                    threadResults[resIdxM * TN + resIdxN] += regM[resIdxM] * regN[resIdxN];
                }
            }
        }
        __syncthreads();
    }

    // 將結果以 float4 向量化寫回 Global Memory
    for (uint resIdxM = 0; resIdxM < TM; ++resIdxM) {
        for (uint resIdxN = 0; resIdxN < TN; resIdxN += 4) {
            int r = threadRow * TM + resIdxM;
            int c = threadCol * TN + resIdxN;
            if ((cRow * BM + r) < M && (cCol * BN + c) < N) {
                // 讀取當前 C 的舊值並寫回
                float4 old_val = reinterpret_cast<const float4*>(&C[r * N + c])[0];
                float4 result;
                result.x = alpha * threadResults[resIdxM * TN + resIdxN + 0] + beta * old_val.x;
                result.y = alpha * threadResults[resIdxM * TN + resIdxN + 1] + beta * old_val.y;
                result.z = alpha * threadResults[resIdxM * TN + resIdxN + 2] + beta * old_val.z;
                result.w = alpha * threadResults[resIdxM * TN + resIdxN + 3] + beta * old_val.w;
                reinterpret_cast<float4*>(&C[r * N + c])[0] = result;
            }
        }
    }
}

int main() {
    int M = 2048, N = 2048, K = 2048;
    size_t size_A = M * K * sizeof(float);
    size_t size_B = K * N * sizeof(float);
    size_t size_C = M * N * sizeof(float);

    float *h_A = (float*)malloc(size_A);
    float *h_B = (float*)malloc(size_B);
    float *h_C = (float*)malloc(size_C);

    for (int i = 0; i < M * K; ++i) h_A[i] = 1.0f;
    for (int i = 0; i < K * N; ++i) h_B[i] = 2.0f;

    float *d_A, *d_B, *d_C;
    cudaMalloc(&d_A, size_A);
    cudaMalloc(&d_B, size_B);
    cudaMalloc(&d_C, size_C);

    cudaMemcpy(d_A, h_A, size_A, cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, h_B, size_B, cudaMemcpyHostToDevice);

    dim3 blockDim((BM * BN) / (TM * TN), 1, 1);
    dim3 gridDim(CEIL_DIV(M, BM), CEIL_DIV(N, BN), 1);

    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    // Warm up
    sgemm_vectorized<<<gridDim, blockDim>>>(M, N, K, 1.0f, d_A, d_B, 0.0f, d_C);
    cudaDeviceSynchronize();

    // 計時
    cudaEventRecord(start);
    sgemm_vectorized<<<gridDim, blockDim>>>(M, N, K, 1.0f, d_A, d_B, 0.0f, d_C);
    cudaEventRecord(stop);
    cudaEventSynchronize(stop);

    float milliseconds = 0;
    cudaEventElapsedTime(&milliseconds, start, stop);

    cudaMemcpy(h_C, d_C, size_C, cudaMemcpyDeviceToHost);
    printf("Verification: C[0] = %f (Expected: %f)\n", h_C[0], (float)K * 2.0f);

    double flops = 2.0 * (double)M * (double)N * (double)K;
    double gflops = (flops / (milliseconds / 1000.0)) / 1e9;
    printf("Execution Time: %.2f ms | Performance: %.2f GFLOPS\n", milliseconds, gflops);

    cudaFree(d_A); cudaFree(d_B); cudaFree(d_C);
    free(h_A); free(h_B); free(h_C);
    return 0;
}

Writing matmul_kernel6.cu


In [ ]:
!nvcc -O3 -arch=sm_75 -lcublas matmul_kernel6.cu -o matmul6
!./matmul6

Verification: C[0] = 4096.000000 (Expected: 4096.000000)
Execution Time: 1.55 ms | Performance: 11074.07 GFLOPS


In [ ]:
%%writefile matmul_final.cu
#include <cstdio>
#include <cstdlib>
#include <cmath>
#include <cuda_runtime.h>
#include <cublas_v2.h>

#define CEIL_DIV(M, N) (((M) + (N) - 1) / (N))

const int BM = 128;
const int BN = 128;
const int BK = 16;
const int WM = 64;
const int WN = 64;
const int WNITER = 4;
const int TN = 4;
const int TM = 8;
const int WARPSIZE = 32;

// Warp-level Tiled SGEMM
__global__ void sgemm_warptiling(int M, int N, int K, float alpha,
                                 const float *A, const float *B,
                                 float beta, float *C) {
    const uint cRow = blockIdx.y;
    const uint cCol = blockIdx.x;

    const uint warpIdx = threadIdx.x / WARPSIZE;
    const uint warpCol = warpIdx % (BN / WN);
    const uint warpRow = warpIdx / (BN / WN);

    const uint threadIdxInWarp = threadIdx.x % WARPSIZE;
    const uint threadColInWarp = threadIdxInWarp % (WN / (TN * WNITER));
    const uint threadRowInWarp = threadIdxInWarp / (WN / (TN * WNITER));

    __shared__ float As[BM * BK];
    __shared__ float Bs[BK * BN];

    A += cRow * BM * K;
    B += cCol * BN;
    C += (cRow * BM + warpRow * WM) * N + (cCol * BN + warpCol * WN);

    const uint innerRowA = threadIdx.x / (BK / 4);
    const uint innerColA = threadIdx.x % (BK / 4);
    const uint strideA = (blockDim.x) / (BK / 4);

    const uint innerRowB = threadIdx.x / (BN / 4);
    const uint innerColB = threadIdx.x % (BN / 4);
    const uint strideB = (blockDim.x) / (BN / 4);

    float threadResults[WM * WN / WARPSIZE] = {0.0f};
    float regM[TM] = {0.0f};
    float regN[TN] = {0.0f};

    for (int bkIdx = 0; bkIdx < K; bkIdx += BK) {
        // 載入 A 到 As
        for (uint offset = 0; offset < BM; offset += strideA) {
            float4 tmp = reinterpret_cast<const float4*>(
                &A[(innerRowA + offset) * K + innerColA * 4])[0];
            As[(innerColA * 4 + 0) * BM + innerRowA + offset] = tmp.x;
            As[(innerColA * 4 + 1) * BM + innerRowA + offset] = tmp.y;
            As[(innerColA * 4 + 2) * BM + innerRowA + offset] = tmp.z;
            As[(innerColA * 4 + 3) * BM + innerRowA + offset] = tmp.w;
        }

        // 載入 B 到 Bs
        for (uint offset = 0; offset < BK; offset += strideB) {
            reinterpret_cast<float4*>(
                &Bs[(innerRowB + offset) * BN + innerColB * 4])[0] =
                reinterpret_cast<const float4*>(&B[(innerRowB + offset) * N + innerColB * 4])[0];
        }
        __syncthreads();

        A += BK;
        B += BK * N;

        // 計算 Warp 內部 Tile 外積
        for (uint dotIdx = 0; dotIdx < BK; ++dotIdx) {
            for (uint wSubRowIdx = 0; wSubRowIdx < WNITER; ++wSubRowIdx) {
                for (uint i = 0; i < TM; ++i) {
                    regM[i] = As[dotIdx * BM + warpRow * WM + threadRowInWarp * TM + i];
                }
                for (uint i = 0; i < TN; ++i) {
                    regN[i] = Bs[dotIdx * BN + warpCol * WN + (wSubRowIdx * (WN / WNITER)) + threadColInWarp * TN + i];
                }
                for (uint resIdxM = 0; resIdxM < TM; ++resIdxM) {
                    for (uint resIdxN = 0; resIdxN < TN; ++resIdxN) {
                        threadResults[(wSubRowIdx * TM + resIdxM) * TN + resIdxN] += regM[resIdxM] * regN[resIdxN];
                    }
                }
            }
        }
        __syncthreads();
    }

    // 寫回 Global Memory
    for (uint wSubRowIdx = 0; wSubRowIdx < WNITER; ++wSubRowIdx) {
        for (uint resIdxM = 0; resIdxM < TM; ++resIdxM) {
            for (uint resIdxN = 0; resIdxN < TN; resIdxN += 4) {
                int r = threadRowInWarp * TM + resIdxM;
                int c = (wSubRowIdx * (WN / WNITER)) + threadColInWarp * TN + resIdxN;
                if ((cRow * BM + warpRow * WM + r) < M && (cCol * BN + warpCol * WN + c) < N) {
                    float4 old_val = reinterpret_cast<const float4*>(&C[r * N + c])[0];
                    float4 result;
                    uint idx = (wSubRowIdx * TM + resIdxM) * TN + resIdxN;
                    result.x = alpha * threadResults[idx + 0] + beta * old_val.x;
                    result.y = alpha * threadResults[idx + 1] + beta * old_val.y;
                    result.z = alpha * threadResults[idx + 2] + beta * old_val.z;
                    result.w = alpha * threadResults[idx + 3] + beta * old_val.w;
                    reinterpret_cast<float4*>(&C[r * N + c])[0] = result;
                }
            }
        }
    }
}

int main() {
    int M = 4096, N = 4096, K = 4096;
    size_t size_A = M * K * sizeof(float);
    size_t size_B = K * N * sizeof(float);
    size_t size_C = M * N * sizeof(float);

    float *h_A = (float*)malloc(size_A);
    float *h_B = (float*)malloc(size_B);
    float *h_C = (float*)malloc(size_C);
    float *h_C_cublas = (float*)malloc(size_C);

    for (int i = 0; i < M * K; ++i) h_A[i] = 1.0f;
    for (int i = 0; i < K * N; ++i) h_B[i] = 2.0f;

    float *d_A, *d_B, *d_C, *d_C_cublas;
    cudaMalloc(&d_A, size_A);
    cudaMalloc(&d_B, size_B);
    cudaMalloc(&d_C, size_C);
    cudaMalloc(&d_C_cublas, size_C);

    cudaMemcpy(d_A, h_A, size_A, cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, h_B, size_B, cudaMemcpyHostToDevice);

    // 4 Warps = 128 Threads per block
    const uint numThreads = (BM / WM) * (BN / WN) * WARPSIZE;
    dim3 blockDim(numThreads, 1, 1);
    dim3 gridDim(CEIL_DIV(N, BN), CEIL_DIV(M, BM), 1);

    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    // 1. 執行客製化 Warp-tiled Kernel
    sgemm_warptiling<<<gridDim, blockDim>>>(M, N, K, 1.0f, d_A, d_B, 0.0f, d_C);
    cudaDeviceSynchronize();

    cudaEventRecord(start);
    sgemm_warptiling<<<gridDim, blockDim>>>(M, N, K, 1.0f, d_A, d_B, 0.0f, d_C);
    cudaEventRecord(stop);
    cudaEventSynchronize(stop);

    float ms_custom = 0;
    cudaEventElapsedTime(&ms_custom, start, stop);

    // 2. 執行 cuBLAS 作為 Baseline 對照
    cublasHandle_t handle;
    cublasCreate(&handle);
    float alpha = 1.0f, beta = 0.0f;

    // Warm up cuBLAS
    cublasSgemm(handle, CUBLAS_OP_N, CUBLAS_OP_N, N, M, K, &alpha, d_B, N, d_A, K, &beta, d_C_cublas, N);
    cudaDeviceSynchronize();

    cudaEventRecord(start);
    cublasSgemm(handle, CUBLAS_OP_N, CUBLAS_OP_N, N, M, K, &alpha, d_B, N, d_A, K, &beta, d_C_cublas, N);
    cudaEventRecord(stop);
    cudaEventSynchronize(stop);

    float ms_cublas = 0;
    cudaEventElapsedTime(&ms_cublas, start, stop);

    // 輸出與比對
    cudaMemcpy(h_C, d_C, size_C, cudaMemcpyDeviceToHost);
    printf("Verification: C[0] = %f (Expected: %f)\n", h_C[0], (float)K * 2.0f);

    double flops = 2.0 * (double)M * (double)N * (double)K;
    double gflops_custom = (flops / (ms_custom / 1000.0)) / 1e9;
    double gflops_cublas = (flops / (ms_cublas / 1000.0)) / 1e9;

    printf("\n================ Performance Comparison (4096 x 4096) ================\n");
    printf("Custom Warp-Tiled Kernel : %7.2f ms | %8.2f GFLOPS\n", ms_custom, gflops_custom);
    printf("NVIDIA cuBLAS (FP32)     : %7.2f ms | %8.2f GFLOPS\n", ms_cublas, gflops_cublas);
    printf("Relative Performance    : %7.2f%%\n", (gflops_custom / gflops_cublas) * 100.0);
    printf("=======================================================================\n");

    cublasDestroy(handle);
    cudaFree(d_A); cudaFree(d_B); cudaFree(d_C); cudaFree(d_C_cublas);
    free(h_A); free(h_B); free(h_C); free(h_C_cublas);
    return 0;
}

Writing matmul_final.cu


In [ ]:
!nvcc -O3 -arch=sm_75 -lcublas matmul_final.cu -o matmul_final
!./matmul_final

Verification: C[0] = 8192.000000 (Expected: 8192.000000)

================ Performance Comparison (4096 x 4096) ================
Custom Warp-Tiled Kernel :   10.42 ms | 13185.75 GFLOPS
NVIDIA cuBLAS (FP32)     :    8.20 ms | 16758.36 GFLOPS
Relative Performance    :   78.68%


In [ ]:
%%writefile matmul_double_buffering.cu
#include <cstdio>
#include <cstdlib>
#include <cmath>
#include <cuda_runtime.h>
#include <cublas_v2.h>

#define CEIL_DIV(M, N) (((M) + (N) - 1) / (N))

// 針對 T4 調優的參數 (BK 改為 8，降低暫存器壓力)
const int BM = 128;
const int BN = 128;
const int BK = 8;
const int TM = 8;
const int TN = 8;

__global__ void sgemm_double_buffering(int M, int N, int K, float alpha,
                                       const float *A, const float *B,
                                       float beta, float *C) {
    const uint cRow = blockIdx.x;
    const uint cCol = blockIdx.y;

    const uint totalResultsBlocktile = BM * BN;
    const uint numThreadsBlocktile = totalResultsBlocktile / (TM * TN); // 256 threads

    const uint threadCol = threadIdx.x % (BN / TN); // 0 ~ 15
    const uint threadRow = threadIdx.x / (BN / TN); // 0 ~ 15

    // Double Buffering: 雙倍 Shared Memory
    __shared__ float As[2][BM * BK];
    __shared__ float Bs[2][BK * BN];

    A += cRow * BM * K;
    B += cCol * BN;
    C += cRow * BM * N + cCol * BN;

    const uint innerRowA = threadIdx.x / (BK / 4);
    const uint innerColA = threadIdx.x % (BK / 4);
    const uint strideA = numThreadsBlocktile / (BK / 4);

    const uint innerRowB = threadIdx.x / (BN / 4);
    const uint innerColB = threadIdx.x % (BN / 4);
    const uint strideB = numThreadsBlocktile / (BN / 4);

    float threadResults[TM * TN] = {0.0f};
    float regM[TM] = {0.0f};
    float regN[TN] = {0.0f};

    // 1. 預先載入第 0 塊 Tile 到 As[0] 與 Bs[0] (Prologue)
    int writeStage = 0;
    for (uint offset = 0; offset < BM; offset += strideA) {
        float4 tmp = reinterpret_cast<const float4*>(&A[(innerRowA + offset) * K + innerColA * 4])[0];
        As[writeStage][(innerRowA + offset) * BK + innerColA * 4 + 0] = tmp.x;
        As[writeStage][(innerRowA + offset) * BK + innerColA * 4 + 1] = tmp.y;
        As[writeStage][(innerRowA + offset) * BK + innerColA * 4 + 2] = tmp.z;
        As[writeStage][(innerRowA + offset) * BK + innerColA * 4 + 3] = tmp.w;
    }
    for (uint offset = 0; offset < BK; offset += strideB) {
        reinterpret_cast<float4*>(&Bs[writeStage][(innerRowB + offset) * BN + innerColB * 4])[0] =
            reinterpret_cast<const float4*>(&B[(innerRowB + offset) * N + innerColB * 4])[0];
    }
    __syncthreads();

    // 2. 主流水線 (Main Loop)
    for (int bkIdx = 0; bkIdx < K - BK; bkIdx += BK) {
        int readStage = writeStage;
        writeStage = writeStage ^ 1; // 切換緩衝區 (0 -> 1, 1 -> 0)

        // 推進 Global 讀取位址
        A += BK;
        B += BK * N;

        // 預先載入下一輪 Tile 到 As[writeStage] / Bs[writeStage] (在暫存器中暫存)
        float4 prefetchA[BM / strideA];
        for (uint i = 0; i < BM / strideA; ++i) {
            uint offset = i * strideA;
            prefetchA[i] = reinterpret_cast<const float4*>(&A[(innerRowA + offset) * K + innerColA * 4])[0];
        }

        float4 prefetchB[BK / strideB];
        for (uint i = 0; i < BK / strideB; ++i) {
            uint offset = i * strideB;
            prefetchB[i] = reinterpret_cast<const float4*>(&B[(innerRowB + offset) * N + innerColB * 4])[0];
        }

        // 同時計算當前緩衝區 (readStage) 的資料
        for (uint dotIdx = 0; dotIdx < BK; ++dotIdx) {
            for (uint i = 0; i < TM; ++i) {
                regM[i] = As[readStage][(threadRow * TM + i) * BK + dotIdx];
            }
            for (uint i = 0; i < TN; ++i) {
                regN[i] = Bs[readStage][dotIdx * BN + (threadCol * TN + i)];
            }
            for (uint resIdxM = 0; resIdxM < TM; ++resIdxM) {
                for (uint resIdxN = 0; resIdxN < TN; ++resIdxN) {
                    threadResults[resIdxM * TN + resIdxN] += regM[resIdxM] * regN[resIdxN];
                }
            }
        }

        // 將預載的資料寫入 As[writeStage] / Bs[writeStage]
        for (uint i = 0; i < BM / strideA; ++i) {
            uint offset = i * strideA;
            As[writeStage][(innerRowA + offset) * BK + innerColA * 4 + 0] = prefetchA[i].x;
            As[writeStage][(innerRowA + offset) * BK + innerColA * 4 + 1] = prefetchA[i].y;
            As[writeStage][(innerRowA + offset) * BK + innerColA * 4 + 2] = prefetchA[i].z;
            As[writeStage][(innerRowA + offset) * BK + innerColA * 4 + 3] = prefetchA[i].w;
        }
        for (uint i = 0; i < BK / strideB; ++i) {
            uint offset = i * strideB;
            reinterpret_cast<float4*>(&Bs[writeStage][(innerRowB + offset) * BN + innerColB * 4])[0] = prefetchB[i];
        }
        __syncthreads();
    }

    // 3. 處理最後一塊 Tile (Epilogue)
    for (uint dotIdx = 0; dotIdx < BK; ++dotIdx) {
        for (uint i = 0; i < TM; ++i) {
            regM[i] = As[writeStage][(threadRow * TM + i) * BK + dotIdx];
        }
        for (uint i = 0; i < TN; ++i) {
            regN[i] = Bs[writeStage][dotIdx * BN + (threadCol * TN + i)];
        }
        for (uint resIdxM = 0; resIdxM < TM; ++resIdxM) {
            for (uint resIdxN = 0; resIdxN < TN; ++resIdxN) {
                threadResults[resIdxM * TN + resIdxN] += regM[resIdxM] * regN[resIdxN];
            }
        }
    }

    // 寫回 Global Memory
    for (uint resIdxM = 0; resIdxM < TM; ++resIdxM) {
        for (uint resIdxN = 0; resIdxN < TN; resIdxN += 4) {
            int r = threadRow * TM + resIdxM;
            int c = threadCol * TN + resIdxN;
            if ((cRow * BM + r) < M && (cCol * BN + c) < N) {
                float4 old_val = reinterpret_cast<const float4*>(&C[r * N + c])[0];
                float4 result;
                result.x = alpha * threadResults[resIdxM * TN + resIdxN + 0] + beta * old_val.x;
                result.y = alpha * threadResults[resIdxM * TN + resIdxN + 1] + beta * old_val.y;
                result.z = alpha * threadResults[resIdxM * TN + resIdxN + 2] + beta * old_val.z;
                result.w = alpha * threadResults[resIdxM * TN + resIdxN + 3] + beta * old_val.w;
                reinterpret_cast<float4*>(&C[r * N + c])[0] = result;
            }
        }
    }
}

int main() {
    int M = 4096, N = 4096, K = 4096;
    size_t size_A = M * K * sizeof(float);
    size_t size_B = K * N * sizeof(float);
    size_t size_C = M * N * sizeof(float);

    float *h_A = (float*)malloc(size_A);
    float *h_B = (float*)malloc(size_B);
    float *h_C = (float*)malloc(size_C);

    for (int i = 0; i < M * K; ++i) h_A[i] = 1.0f;
    for (int i = 0; i < K * N; ++i) h_B[i] = 2.0f;

    float *d_A, *d_B, *d_C, *d_C_cublas;
    cudaMalloc(&d_A, size_A);
    cudaMalloc(&d_B, size_B);
    cudaMalloc(&d_C, size_C);
    cudaMalloc(&d_C_cublas, size_C);

    cudaMemcpy(d_A, h_A, size_A, cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, h_B, size_B, cudaMemcpyHostToDevice);

    dim3 blockDim((BM * BN) / (TM * TN), 1, 1);
    dim3 gridDim(CEIL_DIV(M, BM), CEIL_DIV(N, BN), 1);

    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    // 1. Custom Double Buffering Kernel
    sgemm_double_buffering<<<gridDim, blockDim>>>(M, N, K, 1.0f, d_A, d_B, 0.0f, d_C);
    cudaDeviceSynchronize();

    cudaEventRecord(start);
    sgemm_double_buffering<<<gridDim, blockDim>>>(M, N, K, 1.0f, d_A, d_B, 0.0f, d_C);
    cudaEventRecord(stop);
    cudaEventSynchronize(stop);

    float ms_custom = 0;
    cudaEventElapsedTime(&ms_custom, start, stop);

    // 2. cuBLAS Baseline
    cublasHandle_t handle;
    cublasCreate(&handle);
    float alpha = 1.0f, beta = 0.0f;

    cublasSgemm(handle, CUBLAS_OP_N, CUBLAS_OP_N, N, M, K, &alpha, d_B, N, d_A, K, &beta, d_C_cublas, N);
    cudaDeviceSynchronize();

    cudaEventRecord(start);
    cublasSgemm(handle, CUBLAS_OP_N, CUBLAS_OP_N, N, M, K, &alpha, d_B, N, d_A, K, &beta, d_C_cublas, N);
    cudaEventRecord(stop);
    cudaEventSynchronize(stop);

    float ms_cublas = 0;
    cudaEventElapsedTime(&ms_cublas, start, stop);

    cudaMemcpy(h_C, d_C, size_C, cudaMemcpyDeviceToHost);
    printf("Verification: C[0] = %f (Expected: %f)\n", h_C[0], (float)K * 2.0f);

    double flops = 2.0 * (double)M * (double)N * (double)K;
    double gflops_custom = (flops / (ms_custom / 1000.0)) / 1e9;
    double gflops_cublas = (flops / (ms_cublas / 1000.0)) / 1e9;

    printf("\n================ Performance Comparison (4096 x 4096) ================\n");
    printf("Custom Double Buffering  : %7.2f ms | %8.2f GFLOPS\n", ms_custom, gflops_custom);
    printf("NVIDIA cuBLAS (FP32)     : %7.2f ms | %8.2f GFLOPS\n", ms_cublas, gflops_cublas);
    printf("Relative Performance    : %7.2f%%\n", (gflops_custom / gflops_cublas) * 100.0);
    printf("=======================================================================\n");

    cublasDestroy(handle);
    cudaFree(d_A); cudaFree(d_B); cudaFree(d_C); cudaFree(d_C_cublas);
    free(h_A); free(h_B); free(h_C);
    return 0;
}

Overwriting matmul_double_buffering.cu


In [ ]:
!nvcc -O3 -arch=sm_80 -lcublas matmul_double_buffering.cu -o matmul_db
!./matmul_db

Verification: C[0] = 8192.000000 (Expected: 8192.000000)

================ Performance Comparison (4096 x 4096) ================
Custom Double Buffering  :    9.57 ms | 14367.13 GFLOPS
NVIDIA cuBLAS (FP32)     :    8.39 ms | 16384.00 GFLOPS
Relative Performance    :   87.69%
